## Import

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

## Load data asli

In [2]:
TRAIN_PATH = Path("../data/processed/train.csv")
TEST_PATH = Path("../data/processed/test.csv")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("TRAIN:", train_df.shape)
print("TEST :", test_df.shape)

assert train_df.shape[0] == 64000
assert test_df.shape[0] == 16000

TRAIN: (64000, 8)
TEST : (16000, 8)


## Load KGE embeddings

In [3]:
KGE_DIR = Path("../outputs/kge_features")

train_kge = pd.read_csv(
    KGE_DIR / "train_kge_embeddings.csv"
)

test_kge = pd.read_csv(
    KGE_DIR / "test_kge_embeddings.csv"
)

print("TRAIN KGE:", train_kge.shape)
print("TEST KGE :", test_kge.shape)

assert train_kge.shape == (64000, 100)
assert test_kge.shape == (16000, 100)

TRAIN KGE: (64000, 100)
TEST KGE : (16000, 100)


## Tentukan fitur asli

In [4]:
ORIGINAL_FEATURES = [
    "weight",
    "height",
    "age",
    "gross_motor",
    "fine_motor_adaptive",
    "language",
    "personal_social"
]

TARGET = "dev_status"

assert TARGET not in ORIGINAL_FEATURES
assert len(ORIGINAL_FEATURES) == 7

## Buat feature concatenation

In [5]:
train_original = train_df[ORIGINAL_FEATURES].reset_index(drop=True)
test_original = test_df[ORIGINAL_FEATURES].reset_index(drop=True)

train_kge = train_kge.reset_index(drop=True)
test_kge = test_kge.reset_index(drop=True)

# Pastikan jumlah baris identik
assert len(train_original) == len(train_kge)
assert len(test_original) == len(test_kge)

# Feature concatenation
X_train_final = pd.concat(
    [train_original, train_kge],
    axis=1
)

X_test_final = pd.concat(
    [test_original, test_kge],
    axis=1
)

print("X_train_final:", X_train_final.shape)
print("X_test_final :", X_test_final.shape)

X_train_final: (64000, 107)
X_test_final : (16000, 107)


## Target dipisahkan

In [6]:
y_train = train_df[TARGET].copy()
y_test = test_df[TARGET].copy()

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nDistribusi TRAIN:")
print(y_train.value_counts())

print("\nDistribusi TEST:")
print(y_test.value_counts())

y_train: (64000,)
y_test : (16000,)

Distribusi TRAIN:
dev_status
suspect    51182
normal     12818
Name: count, dtype: int64

Distribusi TEST:
dev_status
suspect    12796
normal      3204
Name: count, dtype: int64


## Tentukan categorical features

In [7]:
CATEGORICAL_FEATURES = [
    "gross_motor",
    "fine_motor_adaptive",
    "language",
    "personal_social"
]

NUMERIC_FEATURES = [
    "weight",
    "height",
    "age"
] + [
    f"kge_{i}"
    for i in range(1, 101)
]

assert len(CATEGORICAL_FEATURES) == 4
assert len(NUMERIC_FEATURES) == 103

assert (
    len(CATEGORICAL_FEATURES)
    + len(NUMERIC_FEATURES)
    == 107
)

print("Categorical:", len(CATEGORICAL_FEATURES))
print("Numerical  :", len(NUMERIC_FEATURES))
print("Total      :", 107)

Categorical: 4
Numerical  : 103
Total      : 107


## Validasi leakage

In [8]:
print("=" * 60)
print("CHECKPOINT 6 — LEAKAGE CHECK")
print("=" * 60)

# Target tidak boleh ada di feature set
assert TARGET not in X_train_final.columns
assert TARGET not in X_test_final.columns

# Jumlah data
assert X_train_final.shape == (64000, 107)
assert X_test_final.shape == (16000, 107)

# Tidak ada missing
assert X_train_final.isna().sum().sum() == 0
assert X_test_final.isna().sum().sum() == 0

# Kolom train dan test harus identik
assert list(X_train_final.columns) == list(X_test_final.columns)

print("✅ TRAIN = 64.000")
print("✅ TEST = 16.000")
print("✅ Total fitur = 107")
print("✅ dev_status tidak masuk feature")
print("✅ Tidak ada missing value")
print("✅ Kolom TRAIN dan TEST identik")

CHECKPOINT 6 — LEAKAGE CHECK
✅ TRAIN = 64.000
✅ TEST = 16.000
✅ Total fitur = 107
✅ dev_status tidak masuk feature
✅ Tidak ada missing value
✅ Kolom TRAIN dan TEST identik


## Simpan hasil

In [9]:
OUTPUT_DIR = Path("../outputs/feature_integration")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

X_train_final.to_csv(
    OUTPUT_DIR / "train_features.csv",
    index=False
)

X_test_final.to_csv(
    OUTPUT_DIR / "test_features.csv",
    index=False
)

y_train.to_csv(
    OUTPUT_DIR / "train_target.csv",
    index=False
)

y_test.to_csv(
    OUTPUT_DIR / "test_target.csv",
    index=False
)

metadata = {
    "train_rows": int(len(X_train_final)),
    "test_rows": int(len(X_test_final)),
    "original_features": ORIGINAL_FEATURES,
    "kge_dimensions": 100,
    "total_features": 107,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "target": TARGET,
    "feature_integration": "concatenation"
}

with open(
    OUTPUT_DIR / "feature_integration_metadata.json",
    "w"
) as f:
    json.dump(
        metadata,
        f,
        indent=4
    )

print("✅ Feature Integration selesai.")

✅ Feature Integration selesai.


## Final check

In [10]:
# Tabel Ringkasan Integrasi Fitur

summary_integration = pd.DataFrame({
    "Komponen": [
        "Fitur asli",
        "Dimensi embedding KGE",
        "Total fitur akhir",
        "Jumlah data latih",
        "Jumlah data uji"
    ],
    "Jumlah": [
        len(ORIGINAL_FEATURES),
        train_kge.shape[1],
        X_train_final.shape[1],
        X_train_final.shape[0],
        X_test_final.shape[0]
    ]
})

summary_integration["Jumlah"] = (
    summary_integration["Jumlah"].map("{:,}".format)
)

display(summary_integration)


,Komponen,Jumlah
0,Fitur asli,7
1,Dimensi embedding KGE,100
2,Total fitur akhir,107
3,Jumlah data latih,"64,000"
4,Jumlah data uji,"16,000"


In [11]:
print("\n" + "=" * 60)
print("CHECKPOINT 6 — FINAL")
print("=" * 60)

print("X_train_final :", X_train_final.shape)
print("X_test_final  :", X_test_final.shape)
print("y_train       :", y_train.shape)
print("y_test        :", y_test.shape)

print("\nJumlah fitur:")
print("Original :", len(ORIGINAL_FEATURES))
print("KGE      :", 100)
print("Total    :", X_train_final.shape[1])

print("\nCategorical features:")
print(CATEGORICAL_FEATURES)

print("\nTarget:")
print(TARGET)

print("\n✅ CHECKPOINT 6 SELESAI")


CHECKPOINT 6 — FINAL
X_train_final : (64000, 107)
X_test_final  : (16000, 107)
y_train       : (64000,)
y_test        : (16000,)

Jumlah fitur:
Original : 7
KGE      : 100
Total    : 107

Categorical features:
['gross_motor', 'fine_motor_adaptive', 'language', 'personal_social']

Target:
dev_status

✅ CHECKPOINT 6 SELESAI
